#### 6.1. Verifying  libraries

In [5]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    try:
        v = version(lib)
        print(f'{lib}: {v}')
    except Exception as e:
        print(f'{lib}: Not found')

reasoning_from_scratch: Not found
torch: 2.4.1+cu124
tokenizers: 0.23.3


#### 6.2. Loading a pretrained model

In [6]:
import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import load_model_and_tokenizer

device = get_device()

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using NVIDIA CUDA GPU
✓ qwen3\qwen3-0.6B-base.pth already up-to-date


In [14]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

 \boxed{21}

#### 6.3. Loading a MATH training subset

In [17]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)

    url = (
        "https://huggingface.co/datasets/"
        "shubhamisok/math-500/resolve/main/"
        "math_full_minus_math500.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        
        
        data = r.json()

        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

math_train = load_math_train()
print("Dataset size:", len(math_train))

Dataset size: 12000


In [18]:
from pprint import pprint
pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

#### 6.4. Sampling rollouts

In [19]:
from reasoning_from_scratch.qwen3 import KVCache
from reasoning_from_scratch.ch04 import top_p_filter


@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for step in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        token_id = next_token.item()
        generated.append(token_id)

        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break
        if step + 1 < max_new_tokens:  # Skips the forward pass after the last token is generated
            logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

In [23]:
raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

 To solve the equation:

\[
\frac{1}{2}(3x - 9) = x + 37
\]

**Step 1: Eliminate the fraction**

Multiply both sides by 2 to eliminate the fraction:

\[
2 \times \frac{1}{2}(3x - 9) = 2 \times (x + 37)
\]

Simplifying both sides:

\[
3x - 9 = 2x + 74
\]

**Step 2: Isolate the variable \( x \)**

Subtract \( 2x \) from both sides:

\[
3x - 2x - 9 = 74
\]

Simplify:

\[
x - 9 = 74
\]

**Step 3: Solve for \( x \)**

Add 9 to both sides:

\[
x = 74 + 9
\]

\[
x = 83
\]

**Final Answer:**

\[
\boxed{83}
\]<|endoftext|>


#### 6.5. Calculating rewards

In [25]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(
        answer_text, fallback=None
    )

    if not extracted:
        return 0.0

    correct = grade_answer(extracted, ground_truth)

    return float(correct)

In [26]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer, "83")
    rollout_rewards.append(reward)
    print(f"{answer!r} -> {reward}")

'\\boxed{83}' -> 1.0
'The correct answer is \\boxed{83}' -> 1.0
'The final answer is 83' -> 0.0
'We get \\boxed{38}' -> 0.0


#### 6.6. Preparing learning signals from rollouts via advantages

In [28]:
rewards = torch.tensor(rollout_rewards, device=device)

advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print("Rewards    :", rewards)
print("Advantages :", advantages)

Rewards    : tensor([1., 1., 0., 0.], device='cuda:0')
Advantages : tensor([ 0.8659,  0.8659, -0.8659, -0.8659], device='cuda:0')


#### 6.7. Scoring rollouts with sequence log-probabilities

In [29]:
@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):

    # Encode prompt and answer tokens separately to get the prompt length later
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)

    # Same as in calc_next_token_logprobas before
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)

    # Index range for positions corresponding to answer tokens
    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1

    # Same as before, except for using start and end
    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Average over the answer token scores
    return torch.mean(next_token_logps).item()

In [30]:
avg_logprob_val = avg_logprob_answer(
                   model, tokenizer, 
                   prompt=prompt,
                   answer=answer_text,
                   device=device) 
print(avg_logprob_val)

-0.0703125


In [32]:
sequence_logprob_val = avg_logprob_val * (len(tokenizer.encode(answer_text)))
print(sequence_logprob_val)

-15.75


In [40]:
def sequence_logprob_draft(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)

    # positions whose next token probas we want
    start = prompt_len - 1
    end = token_ids.shape[0] - 1

    t_idx = torch.arange(start, end, device=token_ids.device)
    next_tokens = token_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # sum over the answer token scores
    return torch.sum(next_token_logps)

print(sequence_logprob_draft(model, token_ids, prompt_len))

tensor(-15.7349, device='cuda:0', grad_fn=<SumBackward0>)


In [41]:
def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(
        1, token_ids[1:].unsqueeze(-1)
    ).squeeze(-1)
    return torch.sum(selected[prompt_len - 1:])

print(sequence_logprob(model, token_ids, prompt_len))

tensor(-15.7349, device='cuda:0', grad_fn=<SumBackward0>)


In [47]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(model, torch.tensor(token_ids, device=device), prompt_len)
    rollout_logps.append(logprob)

    print(f"Answer  : {text}")
    print(f"Logprob : {logprob.item():.4f}")


Answer  : \boxed{83}
Logprob : -8.2359
Answer  : The correct answer is \boxed{83}
Logprob : -19.8304
Answer  : The final answer is 83
Logprob : -16.1606
Answer  : We get \boxed{38}
Logprob : -23.2249


#### 6.8. From advantages to policy updates via the GRPO loss

In [48]:
logps = torch.stack(rollout_logps)
print(logps)

tensor([ -8.2359, -19.8304, -16.1606, -23.2249], device='cuda:0',
       grad_fn=<StackBackward0>)


In [50]:
pg_loss = -(advantages.detach() * logps).mean()

#### 6.9. Putting everything together in a GRPO step

In [51]:
def compute_grpo_loss(
        model, tokenizer, example, device,
        num_rollouts=2, max_new_tokens=256,
        temperature=0.8, top_p=0.9
):
    assert num_rollouts >= 2 
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        # 1. generate rollouts
        token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )

        # 2. compute rewards
        reward = reward_rlvr(answer_text, example["solution"])

        # 3. compute log probs
        logp = sequence_logprob(model, token_ids, prompt_len)

        roll_logps.append(logp)
        roll_rewards.append(reward)
        samples.append(
            {
                "text": text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        model.train()

    # 2. collect all rewards
    rewards = torch.tensor(roll_rewards, device=device)

    # 3. compute advantages
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # collect all logprobs
    logps = torch.stack(roll_logps)

    # compute policy gradient loss
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

In [52]:
stats = compute_grpo_loss(
    model=model,
    tokenizer=tokenizer,
    example=math_train[4],
    device=device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9
)

pprint(stats)

{'advantages': [0.0, 0.0],
 'loss': -0.0,
 'loss_tensor': tensor(-0., device='cuda:0', grad_fn=<NegBackward0>),
 'pg_loss': -0.0,
 'rewards': [0.0, 0.0],
 'samples': [{'gen_len': 256, 'reward': 0.0, 'text': 'We get \\boxed{38}'},
             {'gen_len': 256, 'reward': 0.0, 'text': 'We get \\boxed{38}'}]}


#### 6.10. Batched GRPO setup

In [60]:
@torch.no_grad()
def sample_responses_batched(
    model,
    tokenizer,
    prompt,
    device,
    batch_size,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    prompt_ids = torch.tensor(tokenizer.encode(prompt), device=device, dtype=torch.long)
    prompt_len = prompt_ids.numel()
    input_ids = prompt_ids.unsqueeze(0).expand(batch_size, -1)

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids, cache=cache)[:, -1]

    eos_id = tokenizer.eos_token_id
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    generated_steps = []
    for step in range(max_new_tokens):
        step_logits = logits
        if temperature and temperature != 1.0:
            step_logits = step_logits / temperature

        probas = torch.softmax(step_logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(probas, num_samples=1)

        eos_tok = next_token.new_full((batch_size, 1), eos_id)
        next_token = torch.where(
            finished.view(batch_size, 1), eos_tok, next_token
        )

        generated_steps.append(next_token)

        finished = finished | (next_token.squeeze(1) == eos_id)
        if torch.all(finished):
            break

        if step + 1 < max_new_tokens:  # Skips the forward pass after the last token is generated
            logits = model(next_token, cache=cache)[:, -1]

    if generated_steps:
        gen_tokens = torch.cat(generated_steps, dim=1)
    else:
        gen_tokens = torch.empty((batch_size, 0), dtype=input_ids.dtype, device=device)

    results = []
    for idx in range(batch_size):
        row_tokens = gen_tokens[idx]
        eos_pos = (row_tokens == eos_id).nonzero(as_tuple=True)[0]
        if len(eos_pos) > 0:
            row_tokens = row_tokens[: eos_pos[0] + 1]

        full_token_ids = torch.cat([prompt_ids, row_tokens], dim=0)
        gen_text = tokenizer.decode(row_tokens.tolist())
        results.append((full_token_ids, prompt_len, gen_text))

    return results

In [61]:
def compute_grpo_loss_batched(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=4,
    batch_size=None,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    skip_zero_adv=False,
):
    roll_rewards, samples, rollout_data = [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    if batch_size is None or batch_size <= 0:
        batch_size = num_rollouts

    remaining = num_rollouts
    while remaining > 0:
        current_batch = min(batch_size, remaining)
        batch = sample_responses_batched(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            batch_size=current_batch,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        for token_ids, prompt_len, text in batch:
            reward = reward_rlvr(text, example["answer"])

            roll_rewards.append(reward)
            rollout_data.append((token_ids, prompt_len))
            samples.append(
                {
                    "text": text,
                    "reward": reward,
                    "gen_len": token_ids.numel() - prompt_len,
                }
            )
        remaining -= current_batch

    if was_training:
        model.train()

    rewards = torch.tensor(roll_rewards, device=device)
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    is_zero_adv = torch.allclose(
        advantages,
        torch.zeros_like(advantages),
        atol=1e-8,
        rtol=0.0,
    )

    if skip_zero_adv and is_zero_adv:
        return {
            "loss": 0.0,
            "pg_loss": 0.0,
            "rewards": roll_rewards,
            "advantages": advantages.detach().cpu().tolist(),
            "is_zero_adv": True,
            "samples": samples,
            "loss_tensor": None,
        }

    roll_logps = []
    for token_ids, prompt_len in rollout_data:
        logp = sequence_logprob(model, token_ids, prompt_len)
        roll_logps.append(logp)

    logps = torch.stack(roll_logps)

    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "is_zero_adv": is_zero_adv,
        "samples": samples,
        "loss_tensor": loss,
    }


In [62]:
stats = compute_grpo_loss_batched(
    model=model,
    tokenizer=tokenizer,
    example=math_train[4],
    device=device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9
)

pprint(stats)

{'advantages': [0.0, 0.0],
 'is_zero_adv': True,
 'loss': -0.0,
 'loss_tensor': tensor(-0., device='cuda:0', grad_fn=<NegBackward0>),
 'pg_loss': -0.0,
 'rewards': [0.0, 0.0],
 'samples': [{'gen_len': 4, 'reward': 0.0, 'text': ' 10<|endoftext|>'},
             {'gen_len': 256,
              'reward': 0.0,
              'text': " Let's solve the problem step by step.\n"
                      '\n'
                      '1. **Define Variables:**\n'
                      '   - Let \\( x \\) be the number of days Sam works.\n'
                      '   - Therefore, the number of days he does not work is '
                      '\\( 20 - x \\).\n'
                      '\n'
                      '2. **Calculate Earnings:**\n'
                      '   - Earnings from working days: \\( 60x \\)\n'
                      '   - Earnings from non-working days: \\( 30(20 - x) '
                      '\\)\n'
                      '\n'
                      '3. **Total Earnings:**\n'
                